# ESM C Implementation (TODO; ?% Validation Accuracy, ?% Test Accuracy)

Within each cycle of active learning, you can:

1. Collect training data (original training data + your query data).

2. Train a prediction model to predict the DMS_score for each mutant (e.g., M0A).

3. Use the trained model to predict the score for all mutant in the test set.

4. Select query mutants for next round based on certain criteria. You may want to make sure you don't query the same mutant twice as you only have limited chances to make queries.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, Dataset
import random
from copy import deepcopy
import pandas as pd
from scipy.stats import spearmanr
import argparse
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

import os
from tqdm.auto import tqdm

from esm.models.esmc import ESMC
from esm.sdk.api import ESMProtein, LogitsConfig

## 1. Collect Training Data

Upload `sequence.fasta`, `train.csv`, and `test.csv` to the current runtime:

1. click the folder icon on the left

2. click the upload icon and upload the files to the current directory

In [2]:
#with open('Hackathon_data/Hackathon_data/sequence.fasta', 'r') as f:
with open('sequence.fasta', 'r') as f:
  data = f.readlines()

sequence_wt = data[1].strip()
sequence_wt

'MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLREKMRRRLESGDKWFSLEFFPPRTAEGAVNLISRFDRMAAGGPLYIDVTWHPAGDPGSDKETSSMMIASTAVNYCGLETILHMTCCRQRLEEITGHLHKAKQLGLKNIMALRGDPIGDQWEEEEGGFNYAVDLVKHIRSEFGDYFDICVAGYPKGHPEAGSFEADLKHLKEKVSAGADFIITQLFFEADTFFRFVKACTDMGITCPIVPGIFPIQGYHSLRQLVKLSKLEVPQEIKDVIEPIKDNDAAIRNYGIELAVSLCQELLASGLVPGLHFYTLNREMATTEVLKRLGMWTEDPRRPLPWALSAHPKRREEDVRPIFWASRPKSYIYRTQEWDEFPNGRWGNSSSPAFGELKDYYLFYLKSKSPKEELLKMWGEELTSEESVFEVFVLYLSGEPNRNGHKVTCLPWNDEPLAAETSLLKEELLRVNRQGILTINSQPNINGKPSSDPIVGWGPSGGYVFQKAYLEFFTSRETAEALLQVLKKYELRVNYHLVNVKGENITNAPELQPNAVTWGIFPGREIIQPTVVDPVSFMFWKDEAFALWIERWGKLYEEESPSRTIIQYIHDNYFLVNLVDNDFPLDNCLWQVVEDTLELLNRPTQNARETEAP'

In [3]:
len(sequence_wt)

656

In [4]:
def get_mutated_sequence(mut, sequence_wt):
    '''
    Adds the specified mutation into the wild-type sequence.

    Params:
        mut (str): The mutation to be applied.
        sequence_wt (str): The wild-type sequence to which the mutation will be applied.
    Returns:
        A deep copy of the mutated sequence string.
    '''
    # wt - wild type; pos - position; mt - mutation
    wt, pos, mt = mut[0], int(mut[1:-1]), mut[-1]

    sequence = deepcopy(sequence_wt)

    return sequence[:pos] + mt + sequence[pos+1:]

In [5]:
#df_train = pd.read_csv('Hackathon_data/Hackathon_data/train.csv')
df_train = pd.read_csv('train.csv')
df_train['sequence'] = df_train.mutant.apply(lambda x: get_mutated_sequence(x, sequence_wt))
df_train

,mutant,DMS_score,sequence
0,M0Y,0.2730,YVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1,M0W,0.2857,WVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
2,M0V,0.2153,VVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
3,M0T,0.3122,TVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
4,M0S,0.2180,SVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
...,...,...,...
1135,P347D,0.3876,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1136,P347C,0.1837,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1137,P347A,0.4611,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1138,P347M,0.2412,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...


In [6]:
#df_test = pd.read_csv('Hackathon_data/Hackathon_data/test.csv')
df_test = pd.read_csv('test.csv')
df_test['sequence'] = df_test.mutant.apply(lambda x: get_mutated_sequence(x, sequence_wt))
df_test

,mutant,sequence
0,V1D,MDNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
1,V1Y,MYNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
2,V1C,MCNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
3,V1A,MANEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
4,V1E,MENEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
...,...,...
11319,P655S,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11320,P655T,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11321,P655V,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...
11322,P655A,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...


In [7]:
# TODO: integrate the query data that you acquired each round into df_train

## 2. Train a Prediction Model

Here, we provided a linear regression model and used one-hot encoding to encode each variant. You would need to build your own model to achieve better performances.

Hint: you can perform cross-validation on the training set to evaluate your predictor before making predictions on the test set.

### Hyperparameters

In [8]:
seq_length = 656
seed = 0 # seed for splitting the validation set
val_ratio = 0.2 # proportion of validation set

### ProteinDataset
Convert amino acids to machine-readable data via embedding space used by ESM-2.

In [12]:
def gen_emb_from_df(df, out_dir="esm_c_embeddings_variants", device="cuda:0", batch_size=8):
    '''
    Generate and cache ESM-C sequence embeddings for each unique mutant.
    Saves one mean-pooled embedding tensor per mutant to out_dir/{mutant}.pt.
    '''
    os.makedirs(out_dir, exist_ok=True)

    if isinstance(device, torch.device):
        use_cuda = device.type == "cuda"
    else:
        use_cuda = str(device).startswith("cuda")
    device = torch.device(device) if not isinstance(device, torch.device) else device

    # Each item is (name_for_file, sequence)
    data = [(m, s[:1000]) for m, s in df[["mutant", "sequence"]].drop_duplicates().values]
    print(f"Number of unique variants: {len(data)}")

    # Instantiate 600-million-parameter ESM-C model
    model = ESMC.from_pretrained("esmc_600m").to(device).eval()
    if use_cuda:
        model.half()  # Half-prec. to reduce GPU memory and runtime

    for i in tqdm(range(int(np.ceil(len(data) / batch_size)))):
        batch = data[i * batch_size:(i + 1) * batch_size]

        # Cache skip
        if all(os.path.exists(os.path.join(out_dir, f"{name}.pt")) for name, _ in batch):
            continue

        for name, sequence in batch:
            path = os.path.join(out_dir, f"{name}.pt")
            if os.path.exists(path):
                continue

            protein = ESMProtein(sequence=sequence)

            with torch.no_grad():
                protein_tensor = model.encode(protein)
                model_output = model.logits(
                    protein_tensor,
                    LogitsConfig(sequence=True, return_embeddings=True),
                )

            seq_emb = model_output.embeddings

            # Handle both [L, D] and [1, L, D] output shapes.
            if seq_emb.ndim == 3:
                seq_emb = seq_emb[0]
            seq_mean = seq_emb.float().mean(dim=0).detach().cpu()

            torch.save(seq_mean, path)

class DmsESMDataset(Dataset):
    def __init__(self, df, emb_dir, is_train=True):
        self.df = df.reset_index(drop=True)
        self.emb_dir = emb_dir
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        mutant = self.df.loc[idx, "mutant"]
        emb = torch.load(os.path.join(self.emb_dir, f"{mutant}.pt")).float()
        if self.is_train:
            y = torch.tensor(self.df.loc[idx, "DMS_score"], dtype=torch.float32)
            return emb, y
        return emb, torch.tensor(0.0, dtype=torch.float32)

In [13]:
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("No CUDA device found; using CPU.")

# Run on GPU 0 when available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

True
NVIDIA GeForce RTX 5070 Laptop GPU


In [14]:
# Combine our datasets for embedding, them split them into train / test sets afterward. Each has mutated sequence variants stored as a separate .pt embedding.
all_df = pd.concat(
    [df_train[["mutant", "sequence"]], df_test[["mutant", "sequence"]]],
    ignore_index=True
).drop_duplicates("mutant")

gen_emb_from_df(all_df, out_dir="esm_embeddings_variants", device=device, batch_size=8)

Number of unique variants: 12464


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

c:\Users\Cole\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Cole\.cache\huggingface\hub\models--EvolutionaryScale--esmc-600m-2024-12. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


.gitattributes: 0.00B [00:00, ?B/s]

esmc_600m_2024_12_v0.pth:   0%|          | 0.00/2.30G [00:00<?, ?B/s]

OSError: [WinError 1314] A required privilege is not held by the client: '..\\..\\blobs\\67901bffa44c764b8edc007ffcfaf29d90464e4c' -> 'C:\\Users\\Cole\\.cache\\huggingface\\hub\\models--EvolutionaryScale--esmc-600m-2024-12\\snapshots\\d11cc14d44078eaecbc6a843d5eb20f4eecc1e7e\\README.md'

In [ ]:
# gen_emb('Hackathon_Data/Hackathon_data/sequence.fasta', out_dir='esm_embeddings_train')
# gen_emb('Hackathon_Data/Hackathon_data/', out_dir='esm_embeddings_test')

In [ ]:
# Use simple MLP model to predict from ESM-2 embeddings.
class MLPRegressor(nn.Module):
    def __init__(self, input_dim=1280, hidden_dim=640, dropout_p=0.1):
        super(MLPRegressor, self).__init__()

        # Only need to predict a single fitness score every time.
        output_dim = 1

        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim, output_dim),
            nn.Sigmoid() # Ensures fitness scores in range (0,1).
        )

    def forward(self, x):
        return self.layers(x)

In [ ]:
def train_model_esm(model, train_dataset, val_dataset, epochs=100, batch_size=256, lr=1e-3, patience=10, device='cuda:0'):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)

    # Use MSE loss to handle bounded regression.
    criterion = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    best_val_spearman = -np.inf
    best_ckpt = None
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        train_losses = []
        train_preds = []
        train_targets = []

        for inputs, targets in train_loader:
            inputs = inputs.to(device)
            targets = targets.to(device).float()

            optimizer.zero_grad()
            outputs = model(inputs).squeeze(-1)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())
            train_preds.append(outputs.detach().cpu())
            train_targets.append(targets.detach().cpu())

        train_preds = torch.cat(train_preds).numpy()
        train_targets = torch.cat(train_targets).numpy()
        train_spearman = spearmanr(train_preds, train_targets).statistic

        # Validation
        model.eval()
        val_losses = []
        val_preds = []
        val_targets = []
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs = inputs.to(device)
                targets = targets.to(device).float()
                outputs = model(inputs).squeeze(-1)

                val_losses.append(criterion(outputs, targets).item())
                val_preds.append(outputs.detach().cpu())
                val_targets.append(targets.detach().cpu())

        val_preds = torch.cat(val_preds).numpy()
        val_targets = torch.cat(val_targets).numpy()
        val_spearman = spearmanr(val_preds, val_targets).statistic
        mean_train_loss = float(np.mean(train_losses))
        mean_val_loss = float(np.mean(val_losses))

        if np.isnan(train_spearman):
            train_spearman = 0.0
        if np.isnan(val_spearman):
            val_spearman = -1.0

        print(
            f"Epoch {epoch+1}: "
            f"Train Loss={mean_train_loss:.4f}, Train Spearman={train_spearman:.4f}, "
            f"Val Loss={mean_val_loss:.4f}, Val Spearman={val_spearman:.4f}"
        )

        # Early stopping on validation Spearman (ranking quality)
        if val_spearman > best_val_spearman:
            best_val_spearman = val_spearman
            best_ckpt = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

    if best_ckpt is None:
        best_ckpt = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    return model, best_ckpt

In [ ]:
# Split data into train and validation sets, then form DmsESMDataset() classes from each.
df_train_split, df_val_split = train_test_split(
    df_train, test_size=val_ratio, random_state=seed, shuffle=True
)

train_ds = DmsESMDataset(df_train_split, "esm_embeddings_variants", is_train=True)
val_ds = DmsESMDataset(df_val_split, "esm_embeddings_variants", is_train=True)|
test_ds = DmsESMDataset(df_test, "esm_embeddings_variants", is_train=False)

test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

In [ ]:
# --------------- Train our model ---------------
model_esm = MLPRegressor(input_dim=1280,
                          hidden_dim=640,
                          dropout_p=0.3).to(device)
# TODO Currently using test values to make sure this works.
model_esm, best_ckpt_esm = train_model_esm(
    model_esm,
    train_ds,
    val_ds,
    epochs=150,  # 10
    batch_size=256,
    lr=1e-5,  # 1e-3
    patience=40,  # 5
    device=device,
 )
model_esm.load_state_dict(best_ckpt_esm)


# --------------- Test our model ---------------
model_esm.eval()
preds = []
with torch.no_grad():
    for sequences, _ in test_loader:
        # Inference on the test set
        sequences = sequences.to(device)
        outputs = model_esm(sequences)
        dms_score_pred = outputs.squeeze(-1).cpu().numpy()
        preds.extend(dms_score_pred.tolist())

df_pred = df_test.copy()
df_pred["DMS_score_predicted"] = preds

Epoch 1: Train Loss=0.1196, Train Spearman=0.0123, Val Loss=0.1205, Val Spearman=0.0686
Epoch 2: Train Loss=0.1149, Train Spearman=0.0960, Val Loss=0.1203, Val Spearman=0.1194
Epoch 3: Train Loss=0.1142, Train Spearman=0.1541, Val Loss=0.1197, Val Spearman=0.1588
Epoch 4: Train Loss=0.1120, Train Spearman=0.1290, Val Loss=0.1201, Val Spearman=0.1719
Epoch 5: Train Loss=0.1095, Train Spearman=0.1820, Val Loss=0.1208, Val Spearman=0.1693
Epoch 6: Train Loss=0.1064, Train Spearman=0.1996, Val Loss=0.1209, Val Spearman=0.1878
Epoch 7: Train Loss=0.1051, Train Spearman=0.2142, Val Loss=0.1213, Val Spearman=0.1913
Epoch 8: Train Loss=0.1008, Train Spearman=0.2685, Val Loss=0.1211, Val Spearman=0.2139
Epoch 9: Train Loss=0.1012, Train Spearman=0.2719, Val Loss=0.1212, Val Spearman=0.2520
Epoch 10: Train Loss=0.0971, Train Spearman=0.2995, Val Loss=0.1213, Val Spearman=0.2518
Epoch 11: Train Loss=0.0942, Train Spearman=0.2915, Val Loss=0.1224, Val Spearman=0.2510
Epoch 12: Train Loss=0.0963, T

In [ ]:
# Show current ESM-based predictions.
df_pred[['mutant', 'DMS_score_predicted']].head(n=10)

,mutant,DMS_score_predicted
0,V1D,0.486079
1,V1Y,0.248332
2,V1C,0.350649
3,V1A,0.695609
4,V1E,0.613285
5,V1W,0.469172
6,V1T,0.454575
7,V1R,0.388013
8,V1Q,0.480368
9,V1S,0.503567


In [ ]:
# Save predictions to .csv.
df_pred[['mutant', 'DMS_score_predicted']].to_csv('test_predictions.csv', index=False)

In [ ]:
group_number = "1"

with open("GroupName.txt", "w") as f:
    f.write(group_number.strip() + "\n")

print("Saved GroupName.txt")

Saved GroupName.txt


In [ ]:
api_key = "df303a548bec1afcff7d7196650c5396cfa74c94a8be5357043602dcc7537ba7"

with open("APIKey.txt", "w") as f:
    f.write(api_key.strip() + "\n")

print("Saved APIKey.txt")

Saved APIKey.txt


## 3. Select Query for Next Round

In [ ]:
# Show prediction distribution.
df_test.sort_values('DMS_score_predicted', ascending=False)

,mutant,sequence,DMS_score_predicted
280,E15N,MVNEARGNSSLNPCLNGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.310153
88,R5N,MVNEANGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.308870
4661,D281Q,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.308391
8994,L533Q,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.308002
7160,L436K,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.307960
...,...,...,...
4374,R265D,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.283032
4506,K272D,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.282828
2766,R156V,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.282064
2767,R156N,MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...,0.281676


In [ ]:
# Write mutants with top ten DMS scores to .txt file.
df_top_ten = df_pred.sort_values('DMS_score_predicted', ascending=False).head(n=10)
df_top_ten.to_csv('top10.txt', columns=['mutant'], index=False, header=False)
print(df_top_ten)

      mutant                                           sequence  \
280     E15N  MVNEARGNSSLNPCLNGSASSGSESSKDSSRCSTPGLDPERHERLR...   
88       R5N  MVNEANGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...   
4661   D281Q  MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...   
8994   L533Q  MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...   
7160   L436K  MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...   
4655   D281A  MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...   
7136   V435K  MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...   
3907   E231S  MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...   
10131  R593Q  MVNEARGNSSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...   
136      S8G  MVNEARGNGSLNPCLEGSASSGSESSKDSSRCSTPGLDPERHERLR...   

       DMS_score_predicted  
280               0.310153  
88                0.308870  
4661              0.308391  
8994              0.308002  
7160              0.307960  
4655              0.307910  
7136              0.307733  
3907              0.307652  
1

In [ ]:
# Example: randomly select 100 test variants to be queried.
# Note: Random selection may not be a good strategy
# TODO: Select query mutants for the next round based on your own criteria

queries = np.random.choice(df_test.mutant.values, size=100, replace=False)
queries

array(['N117F', 'L438D', 'N11M', 'M153F', 'I137A', 'F79K', 'H95K',
       'N614P', 'S8I', 'A349K', 'V361S', 'E651D', 'K529N', 'P453D',
       'Y403F', 'N2Q', 'E636Y', 'K530C', 'W590K', 'V175I', 'E42D',
       'W338Y', 'R472G', 'C129V', 'V430K', 'L617W', 'Q508V', 'G595Q',
       'T463A', 'K529E', 'D360F', 'K287A', 'E601I', 'E601G', 'S303Q',
       'A649E', 'D628C', 'T226Y', 'T33I', 'Q484I', 'E168P', 'S368Q',
       'H212G', 'E70L', 'Y598P', 'H95C', 'K145P', 'K269A', 'A557W',
       'A523N', 'K529A', 'H200K', 'S602D', 'I591Y', 'L350M', 'G98N',
       'P645C', 'D360E', 'K240N', 'W94I', 'Q277F', 'E53P', 'I124F',
       'K240E', 'A367Y', 'W632Y', 'D103P', 'T114C', 'A557V', 'T33M',
       'N74W', 'A154Y', 'L213A', 'W594T', 'P412V', 'K145C', 'D456A',
       'V449R', 'V178C', 'D209T', 'G503S', 'F616E', 'V116E', 'A232W',
       'K280G', 'R237Y', 'E165G', 'A194T', 'Q132N', 'N455K', 'V449Q',
       'D176V', 'K272G', 'E122D', 'E468Y', 'N546T', 'P285C', 'K509G',
       'I90H', 'P571A'], dtype=objec

In [ ]:
with open('query.txt', 'w') as f:
  for mutant in queries:
    f.write(mutant+'\n')